# 🕵️‍♂️ **Where's Waldo? Train an AI Model to Find Waldo** 🔎

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kaopanboonyuen/WALDO2026/blob/main/code/WALDO_AI_NSTDA2026_toStudent.ipynb)
![Runs on Colab T4 GPU](https://img.shields.io/badge/Colab-T4%20GPU%20Free-orange?logo=googlecolab)
![Ultralytics](https://img.shields.io/badge/Ultralytics-YOLOv8%20%7C%20YOLO11%20%7C%20YOLO12-blueviolet)

### NSTDA AI Workshop 2026 · by Dr.Teerapong Panboonyuen (P'Kao)

Somewhere in a crowd of thousands of tiny cartoon people, **Waldo** is hiding. Today, instead of scanning the page with your own eyes for ten minutes, you'll teach a **YOLO object detector** to do it in **milliseconds**.

### 🗺️ Today's Roadmap

| Step | What we'll do |
|---|---|
| 0 | Set up the environment (GPU check, install `ultralytics`) |
| 1 | Download & organize the Waldo dataset |
| 2 | **EDA** — understand the data *before* we train anything |
| 3 | Configure the YOLO dataset config (`data.yaml`) |
| 4 | Train **3 YOLO architectures** (YOLOv8n → YOLO11n → YOLOv12n) |
| 5 | 🎯 **TODO for you**: add a 4th model of your own |
| 6 | Build a **Scoreboard** comparing every model |
| 7 | Visualize the battle — bar charts, training curves, radar chart, confusion matrices, speed-vs-accuracy |
| 8 | 🏆 Crown the **Winner** |
| 9 | Fun finale — unleash the winner model on a brand-new test image |
| 10 | 🎓 **Mini-Hackathon assignment** (today's homework) |

> 💡 **Note on reproducibility:** Every training run below is given an **explicit name** (e.g. `yolov8n_baseline`), and every path used later is **resolved dynamically** by scanning the `runs/` folder — no hardcoded `train`, `train2`, `train3`... you can re-run any cell as many times as you like and the notebook will always find the right (latest) result automatically.


---
## 📌 Step 0: Environment Setup

We check the GPU, install `ultralytics` (which bundles YOLOv8, YOLO11, and YOLOv12), and import everything we need.

> On Colab: go to **Runtime → Change runtime type → T4 GPU** (free tier) before running the notebook.


In [ ]:
# Install the Ultralytics package (bundles YOLOv8 / YOLO11 / YOLOv12) + light extras for nicer plots
!pip install -q ultralytics plotly kaleido
print("✅ Packages installed.")

In [ ]:
# Drop Your AI Python Stack Here 🔥

In [ ]:
import os
import sys
import glob
import time
import json
import shutil
import zipfile
import random
import warnings
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import Image as IPImage, display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("✅ Imports ready. Random seed set to", SEED)

In [ ]:
import ultralytics

print("🖥️  Environment check")
print("-" * 40)
print(f"Python          : {sys.version.split()[0]}")
print(f"PyTorch         : {torch.__version__}")
print(f"Ultralytics     : {ultralytics.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU name        : {torch.cuda.get_device_name(0)}")
    print(f"GPU count       : {torch.cuda.device_count()}")
    DEVICE = 0
else:
    print("⚠️  No GPU detected — go to Runtime > Change runtime type > T4 GPU for a MUCH faster workshop!")
    DEVICE = "cpu"
print("-" * 40)

---
## 📌 Step 1: Download & Prepare the Dataset

The **Where's Waldo** dataset has 2 classes: `waldo` and `wenda`, split into `train` / `valid` folders in standard YOLO format (`images/` + `labels/*.txt`).

The dataset is hosted as a **multi-part zip** on GitHub (to dodge GitHub's file-size limits). The cell below is **idempotent** — safe to re-run; it skips any step that's already done, so re-running the notebook (or a single cell) never re-downloads or re-extracts unnecessarily.


In [ ]:
DATA_ROOT = # Enter Your Dataset Path Here
RAW_DIR = os.path.join(DATA_ROOT, "waldo-dataset-b")
DATASET_DIR = os.path.join(DATA_ROOT, "dataset")
ZIP_PATH = os.path.join(DATA_ROOT, "waldo-dataset-b.zip")

BASE_URL = "https://github.com/kaopanboonyuen/where-is-waldo/raw/main/dataset"
ZIP_PARTS = [f"waldo-dataset-b.zip.part{i}" for i in range(1, 6)]

os.makedirs(DATA_ROOT, exist_ok=True)

# 1) Download each part only if it isn't already on disk
for part in ZIP_PARTS:
    part_path = os.path.join(DATA_ROOT, part)
    if os.path.exists(part_path):
        print(f"⏭️  {part} already downloaded, skipping.")
        continue
    print(f"⬇️  Downloading {part} ...")
    !wget -q "{BASE_URL}/{part}" -O "{part_path}"

print("✅ All dataset parts present.")

In [ ]:
def reassemble_zip(parts, output_file):
    '''Concatenate zip parts back into a single .zip file (skips if already done).'''
    if os.path.exists(output_file):
        print(f"⏭️  {output_file} already reassembled, skipping.")
        return
    with open(output_file, "wb") as out:
        for part in parts:
            with open(part, "rb") as f:
                out.write(f.read())
    print(f"✅ Reassembled -> {output_file}")

part_paths = [os.path.join(DATA_ROOT, p) for p in ZIP_PARTS]
reassemble_zip(part_paths, ZIP_PATH)

if not os.path.exists(RAW_DIR):
    print("📦 Extracting dataset ...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(DATA_ROOT)
    print("✅ Extracted to", RAW_DIR)
else:
    print("⏭️  Dataset already extracted, skipping.")

In [ ]:
# Organize into the standard YOLO layout: dataset/{train,valid}/{images,labels}
os.makedirs(DATASET_DIR, exist_ok=True)

for split in ["train", "valid"]:
    dst_images = os.path.join(DATASET_DIR, split, "images")
    dst_labels = os.path.join(DATASET_DIR, split, "labels")
    src_images = os.path.join(RAW_DIR, split, "images")
    src_labels = os.path.join(RAW_DIR, split, "labels")

    if os.path.exists(dst_images) and os.path.exists(dst_labels):
        print(f"⏭️  {split} split already organized, skipping.")
        continue

    os.makedirs(os.path.join(DATASET_DIR, split), exist_ok=True)
    shutil.move(src_images, dst_images)
    shutil.move(src_labels, dst_labels)
    print(f"✅ Organized {split} split -> {dst_images}, {dst_labels}")

for split in ["train", "valid"]:
    n_img = len(os.listdir(os.path.join(DATASET_DIR, split, "images")))
    n_lbl = len(os.listdir(os.path.join(DATASET_DIR, split, "labels")))
    print(f"{split:>6}: {n_img} images, {n_lbl} labels")

---
## 📌 Step 2: Exploratory Data Analysis (EDA) — Understand the Problem First!

Before training anything, let's actually *look* at what we're asking the model to do. We'll check:

1. How many images / labels per split
2. **Class balance** — is `waldo` rarer than `wenda`?
3. **Bounding-box size** — this is the crux of the challenge: Waldo is *tiny* relative to the image!
4. Image resolutions in the dataset
5. A visual sample grid with ground-truth boxes overlaid

> 🧠 **Why this matters:** small, rare objects in large images are one of the hardest cases for any detector. Seeing this in the EDA sets up *why* we'll compare several architectures later — some handle tiny objects better than others.


In [ ]:
def collect_label_stats(images_dir, labels_dir):
    '''Parse every YOLO-format label file and return a tidy DataFrame of one row per bounding box.'''
    rows = []
    for label_file in os.listdir(labels_dir):
        if not label_file.endswith(".txt"):
            continue
        img_candidates = [label_file.replace(".txt", ext) for ext in [".jpg", ".jpeg", ".png"]]
        img_file = next((f for f in img_candidates if os.path.exists(os.path.join(images_dir, f))), None)
        if img_file is None:
            continue
        img = cv2.imread(os.path.join(images_dir, img_file))
        if img is None:
            continue
        h, w = img.shape[:2]

        with open(os.path.join(labels_dir, label_file)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                cls_id, xc, yc, bw, bh = int(parts[0]), *map(float, parts[1:5])
                box_w_px, box_h_px = bw * w, bh * h
                rows.append({
                    "image": img_file,
                    "class_id": cls_id,
                    "img_w": w, "img_h": h,
                    "box_w_px": box_w_px, "box_h_px": box_h_px,
                    "box_area_px": box_w_px * box_h_px,
                    "box_area_frac": bw * bh,  # fraction of the image area
                    "aspect_ratio": box_w_px / (box_h_px + 1e-6),
                })
    return pd.DataFrame(rows)

train_stats = collect_label_stats(
    os.path.join(DATASET_DIR, "train", "images"),
    os.path.join(DATASET_DIR, "train", "labels"),
)
valid_stats = collect_label_stats(
    os.path.join(DATASET_DIR, "valid", "images"),
    os.path.join(DATASET_DIR, "valid", "labels"),
)
train_stats["split"] = "train"
valid_stats["split"] = "valid"
all_stats = pd.concat([train_stats, valid_stats], ignore_index=True)

print(f"Total labeled boxes: {len(all_stats)}  (train={len(train_stats)}, valid={len(valid_stats)})")
all_stats.head()

In [ ]:
# EDA Plot 1: images per split
split_counts = pd.DataFrame({
    "split": ["train", "valid"],
    "images": [
        len(os.listdir(os.path.join(DATASET_DIR, "train", "images"))),
        len(os.listdir(os.path.join(DATASET_DIR, "valid", "images"))),
    ],
})

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=split_counts, x="split", y="images", hue="split", legend=False, ax=ax, palette="crest")
for i, v in enumerate(split_counts["images"]):
    ax.text(i, v + max(split_counts["images"]) * 0.02, str(v), ha="center", fontweight="bold")
ax.set_title("📊 Images per Split")
ax.set_ylabel("# images")
plt.tight_layout()
plt.show()

In [ ]:
# EDA Plot 2: class balance (instance-level, not image-level)
CLASS_NAMES = # 🚀 Write Your Code Here
all_stats["class_name"] = all_stats["class_id"].map(CLASS_NAMES)

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(data=all_stats, x="class_name", hue="class_name", legend=False, ax=ax, palette="flare")
ax.set_title("📊 Class Balance — Bounding Box Instances")
ax.set_xlabel("class")
ax.set_ylabel("# instances")
plt.tight_layout()
plt.show()

print(all_stats["class_name"].value_counts(normalize=True).round(3).to_dict())

In [ ]:
# EDA Plot 3: bounding-box size -- this is *the* key insight for Where's Waldo
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

sns.histplot(all_stats["box_area_frac"] * 100, bins=30, kde=True, ax=axes[0], color="#e0546a")
axes[0].set_title("Waldo/Wenda box area\n(% of full image)")
axes[0].set_xlabel("box area (% of image)")

sns.scatterplot(data=all_stats, x="box_w_px", y="box_h_px", hue="class_name", alpha=0.6, ax=axes[1])
axes[1].set_title("Box width vs height (px)")
axes[1].set_xlabel("width (px)")
axes[1].set_ylabel("height (px)")

sns.histplot(all_stats["aspect_ratio"], bins=30, kde=True, ax=axes[2], color="#4c72b0")
axes[2].set_title("Aspect ratio (w / h)")
axes[2].set_xlabel("aspect ratio")

plt.tight_layout()
plt.show()

median_pct = (all_stats["box_area_frac"] * 100).median()
print(f"🔎 Median target size: only ~{median_pct:.2f}% of the image area.")
print("   -> Waldo really is a needle in a haystack. Small-object detection is hard!")

In [ ]:
# EDA Plot 4: source image resolutions in the dataset
res_df = all_stats.drop_duplicates(subset=["image", "split"])[["img_w", "img_h"]]

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(data=res_df, x="img_w", y="img_h", alpha=0.5, ax=ax, color="#55a868")
ax.set_title("📊 Source Image Resolutions")
ax.set_xlabel("width (px)")
ax.set_ylabel("height (px)")
plt.tight_layout()
plt.show()

print(res_df.describe().round(1))

In [ ]:
def show_random_label_overlay(images_dir, labels_dir, num_images=8, class_names=CLASS_NAMES):
    '''Show a grid of random training images with their YOLO ground-truth boxes overlaid.'''
    all_images = [f for f in os.listdir(images_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    chosen = random.sample(all_images, min(num_images, len(all_images)))

    cols = 4
    rows = int(np.ceil(len(chosen) / cols))
    fig, axs = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axs = np.array(axs).reshape(-1)

    colors = {0: (255, 0, 0), 1: (0, 128, 255)}

    for ax, image_file in zip(axs, chosen):
        image = cv2.cvtColor(cv2.imread(os.path.join(images_dir, image_file)), cv2.COLOR_BGR2RGB)
        label_path = os.path.join(labels_dir, os.path.splitext(image_file)[0] + ".txt")
        h, w = image.shape[:2]

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cls_id = int(parts[0])
                    xc, yc, bw, bh = map(float, parts[1:5])
                    x1, y1 = int((xc - bw / 2) * w), int((yc - bh / 2) * h)
                    x2, y2 = int((xc + bw / 2) * w), int((yc + bh / 2) * h)
                    color = colors.get(cls_id, (0, 255, 0))
                    cv2.rectangle(image, (x1, y1), (x2, y2), color, 3)
                    cv2.putText(image, class_names.get(cls_id, str(cls_id)), (x1, max(y1 - 8, 0)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

        ax.imshow(image)
        ax.axis("off")

    for ax in axs[len(chosen):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

show_random_label_overlay(
    os.path.join(DATASET_DIR, "train", "images"),
    os.path.join(DATASET_DIR, "train", "labels"),
)

> 📝 **EDA takeaways:**
> - The dataset is modestly sized — great for a 1-day workshop, but expect some variance between runs.
> - Waldo/Wenda occupy a **tiny fraction** of each image → this is fundamentally a *small-object detection* problem.
> - This is exactly why we'll compare multiple YOLO generations next: newer architectures often handle small objects and class imbalance differently.


---
## 📌 Step 3: Configure the YOLO Dataset (`data.yaml`)

YOLO needs a small YAML file telling it where the images live and what the classes are called.


In [ ]:
DATA_YAML_PATH = # 📂 Set Your Dataset Path Here

data_yaml = f'''
train: {os.path.join(DATASET_DIR, "train", "images")}
val: {os.path.join(DATASET_DIR, "valid", "images")}

nc: 2
names: ['waldo', 'wenda']
'''

with open(DATA_YAML_PATH, "w") as f:
    f.write(data_yaml)

print(f"✅ Wrote {DATA_YAML_PATH}\n")
print(data_yaml)

---
## 📌 Step 4: The Model Zoo — Train & Compare 3 YOLO Architectures

We'll train **three** generations of Ultralytics YOLO on the exact same data, epochs, and image size, so the comparison is fair:

| Model | Released | Idea |
|---|---|---|
| **YOLOv8n** | 2023 | The battle-tested classic. Our baseline. |
| **YOLO11n** | 2024 | Ultralytics' redesigned, more efficient successor to v8. |
| **YOLOv12n** | 2025 | Attention-centric architecture — matches the notebook's original title! |

All three use the **`n` (nano)** size so training stays fast and free-tier-GPU-friendly.

### 🔧 The "no more `train2`, `train3`" trick

Every experiment below gets an **explicit `name=`**, so its results always land in a predictable folder like `runs/detect/yolov8n_baseline`. If you re-run a cell, Ultralytics will auto-append `2`, `3`, ... — so instead of hardcoding a path, we use `get_latest_run_dir()` below, which **dynamically** finds the most recent matching folder. Every later cell (scoreboard, plots, inference) calls this helper — **you never need to hand-edit a path again.**


In [ ]:
PROJECT_DIR = # 📂 Set Your Dataset Path Here

# 🧪 Add / edit experiments here. This list drives everything below.
EXPERIMENTS = [
    # {# 🤖 Design Your Vision-AI Model Here}
]


def get_latest_run_dir(project, name):
    '''Dynamically find the most recent run folder for `name`
    (handles Ultralytics' auto-incrementing name, name2, name3, ... suffix).
    No hardcoded paths, ever.'''
    candidates = glob.glob(os.path.join(project, f"{name}*"))
    candidates = [c for c in candidates if os.path.isdir(c)]
    if not candidates:
        raise FileNotFoundError(f"No run directory found for '{name}' in {project}")
    return max(candidates, key=os.path.getmtime)


def train_experiment(cfg, data_yaml=DATA_YAML_PATH, device=DEVICE, project=PROJECT_DIR):
    '''Train one experiment and return everything the scoreboard needs.'''
    print(f"\n🚀 Training '{cfg['name']}' ({cfg['weights']}) ...")
    t0 = time.time()

    model = # 🚀 Write Your Code Here
    model.train(
        data=data_yaml,
        epochs=cfg["epochs"],
        imgsz=cfg["imgsz"],
        batch=cfg["batch"],
        device=device,
        project=project,
        name=cfg["name"],
        exist_ok=False,
        verbose=False,
        **cfg.get("params", {}),
    )
    elapsed = time.time() - t0

    run_dir = get_latest_run_dir(project, cfg["name"])
    weights_path = os.path.join(run_dir, "weights", "best.pt")
    best_model = # 🚀 Write Your Code Here
    val_metrics = best_model.val(data=data_yaml, verbose=False)

    weights_mb = os.path.getsize(weights_path) / (1024 ** 2)

    return {
        "name": cfg["name"],
        "weights_family": cfg["weights"].replace(".pt", ""),
        "run_dir": run_dir,
        "weights_path": weights_path,
        "model": best_model,
        "train_seconds": elapsed,
        "weights_mb": weights_mb,
        "precision": float(val_metrics.box.mp),
        "recall": float(val_metrics.box.mr),
        "map50": float(val_metrics.box.map50),
        "map50_95": float(val_metrics.box.map),
        "inference_ms": float(val_metrics.speed.get("inference", np.nan)),
    }

print(f"✅ {len(EXPERIMENTS)} experiment(s) configured:")
for e in EXPERIMENTS:
    print("  -", e["name"], "->", e["weights"])

In [ ]:
# 🏋️ Train all configured experiments. This is the slow cell — grab a coffee ☕
# (On a free Colab T4, ~30 epochs on this dataset for one nano model is only a few minutes.)
RUNS = {}

for cfg in EXPERIMENTS:
    result = train_experiment(cfg)
    RUNS[cfg["name"]] = result
    print(f"✅ '{cfg['name']}' done in {result['train_seconds']/60:.1f} min "
          f"| mAP50-95={result['map50_95']:.3f} | run_dir={result['run_dir']}")

print("\n🎉 All experiments finished:", list(RUNS.keys()))

---
## 📌 Step 5: 🎯 TODO — Add Your Own 4th Model!

Now it's your turn. Pick **one** idea below (or invent your own) and add a 4th entry to `EXPERIMENTS`, then train just that one:

1. **Bigger model** — try `yolov8s.pt`, `yolo11s.pt`, or `yolo12s.pt` (more parameters, possibly higher accuracy, slower).
2. **Longer training** — bump `epochs` to 50–60.
3. **Hyperparameter tuning** — pass extra `params`, e.g. `{"lr0": 0.01, "optimizer": "AdamW", "mosaic": 1.0}`.
4. **Different image size** — try `imgsz=960` since Waldo is *tiny* (more resolution can help small objects!).

Fill in the cell below (it's intentionally left blank) 👇


In [ ]:
# ✏️ YOUR TURN — define your 4th experiment and train it.
#
# my_experiment = {"name": "my_custom_model", "weights": "___.pt", "epochs": ___, "imgsz": ___, "batch": 16, "params": {}}
# EXPERIMENTS.append(my_experiment)
# RUNS[my_experiment["name"]] = train_experiment(my_experiment)

# TODO: write your code here


---
## 📌 Step 6: The Scoreboard 🏅

Let's line every trained model up side-by-side. This table is built **entirely from `RUNS`**, which was populated dynamically above — add a 5th, 6th, 10th model and this table (and every plot after it) updates automatically.


In [ ]:
def build_scoreboard(runs: dict) -> pd.DataFrame:
    rows = []
    for name, r in runs.items():
        precision, recall = r["precision"], r["recall"]
        f1 = 2 * precision * recall / (precision + recall + 1e-9)
        rows.append({
            "model": name,
            "architecture": r["weights_family"],
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "mAP50": r["map50"],
            "mAP50-95": r["map50_95"],
            "inference_ms": r["inference_ms"],
            "weights_MB": r["weights_mb"],
            "train_min": r["train_seconds"] / 60,
        })
    df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False).reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)
    return df

scoreboard = # 🚀 Write Your Code Here
scoreboard

In [ ]:
(
    scoreboard.style
    .background_gradient(subset=["precision", "recall", "f1", "mAP50", "mAP50-95"], cmap="Greens")
    .background_gradient(subset=["inference_ms", "weights_MB", "train_min"], cmap="Reds_r")
    .format({
        "precision": "{:.3f}", "recall": "{:.3f}", "f1": "{:.3f}",
        "mAP50": "{:.3f}", "mAP50-95": "{:.3f}",
        "inference_ms": "{:.1f}", "weights_MB": "{:.1f}", "train_min": "{:.1f}",
    })
)

---
## 📌 Step 7: Visualize the Battle 📊

A metrics table is useful — but plots make the story click instantly. Let's look at this from five angles.


In [ ]:
# 7.1 — Accuracy bars: mAP50 vs mAP50-95
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(scoreboard))
width = 0.35

ax.bar(x - width / 2, scoreboard["mAP50"], width, label="mAP50", color="#4c72b0")
ax.bar(x + width / 2, scoreboard["mAP50-95"], width, label="mAP50-95", color="#dd8452")

ax.set_xticks(x)
ax.set_xticklabels(scoreboard["model"], rotation=20, ha="right")
ax.set_ylabel("score")
ax.set_title("📈 Accuracy by Model: mAP50 vs mAP50-95")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7.2 — Precision / Recall / F1 grouped bars
metrics_long = scoreboard.melt(
    id_vars="model", value_vars=["precision", "recall", "f1"],
    var_name="metric", value_name="score",
)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=metrics_long, x="model", y="score", hue="metric", ax=ax, palette="Set2")
ax.set_title("📈 Precision / Recall / F1 by Model")
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# 7.3 — Training curves overlaid: mAP50-95 vs epoch, for every model
# Reads results.csv from each run's dynamically-resolved directory. No hardcoded paths.
fig, ax = plt.subplots(figsize=(9, 5))

for name, r in RUNS.items():
    csv_path = os.path.join(r["run_dir"], "results.csv")
    if not os.path.exists(csv_path):
        continue
    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]
    map_col = next((c for c in df.columns if "mAP50-95" in c and "val" not in c.lower()), None)
    if map_col is None:
        continue
    ax.plot(df["epoch"], df[map_col], marker="o", markersize=3, label=name)

ax.set_xlabel("epoch")
ax.set_ylabel("mAP50-95")
ax.set_title("📈 Training Curves — mAP50-95 over Epochs")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 7.4 — Radar chart: normalized multi-metric comparison (interactive, Plotly)
radar_metrics = # 🚀 Write Your Code Here
norm = scoreboard.copy()
for m in radar_metrics:
    lo, hi = norm[m].min(), norm[m].max()
    norm[m] = 1.0 if hi == lo else (norm[m] - lo) / (hi - lo)

# Efficiency: faster & smaller is better, so invert-normalize before adding to the radar
for m in ["inference_ms", "weights_MB"]:
    lo, hi = scoreboard[m].min(), scoreboard[m].max()
    norm[m] = 1.0 if hi == lo else 1 - (scoreboard[m] - lo) / (hi - lo)

radar_axes = radar_metrics + ["inference_ms", "weights_MB"]
axis_labels = ["Precision", "Recall", "mAP50", "mAP50-95", "Speed (↑=faster)", "Compact (↑=smaller)"]

fig = go.Figure()
for _, row in norm.iterrows():
    values = [row[m] for m in radar_axes]
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],
        theta=axis_labels + [axis_labels[0]],
        fill="toself",
        name=row["model"],
    ))

fig.update_layout(
    title="🕸️ Multi-Metric Radar — Accuracy vs Speed vs Size (normalized 0-1)",
    polar=dict(radialaxis=dict(visible=True, range=[0, 1])),
    showlegend=True,
    width=750, height=600,
)
fig.show()

In [ ]:
# 7.5 — Confusion matrices side-by-side (dynamic paths via RUNS[...]['run_dir'])
n = len(RUNS)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
if n == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, RUNS.items()):
    cm_path = os.path.join(r["run_dir"], "confusion_matrix.png")
    if os.path.exists(cm_path):
        img = plt.imread(cm_path)
        ax.imshow(img)
        ax.set_title(name)
    else:
        ax.text(0.5, 0.5, "confusion_matrix.png\nnot found", ha="center", va="center")
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# 7.6 — Bubble chart: the classic accuracy-vs-speed-vs-size tradeoff, all in one plot
fig = px.scatter(
    scoreboard, x="inference_ms", y="mAP50-95",
    size="weights_MB", color="model", text="model",
    size_max=50,
    labels={"inference_ms": "Inference time (ms/image, lower=better)", "mAP50-95": "mAP50-95 (higher=better)"},
    title="⚖️ Accuracy vs Speed vs Model Size (bubble size = weights MB)",
)
fig.update_traces(textposition="top center")
fig.update_layout(width=800, height=550)
fig.show()

---
## 📌 Step 8: 🏆 Announcing the Winner!

We combine accuracy and efficiency into one **composite score**, weighted toward accuracy (since this is a detection-quality workshop) but still rewarding fast, compact models:

`composite = 0.5·mAP50-95 + 0.2·mAP50 + 0.15·F1 + 0.1·speed_norm + 0.05·size_norm`

Feel free to change the weights and re-run — that's part of the fun of building your own scoreboard!


In [ ]:
composite = # 🚀 Write Your Code Here

def minmax_norm(s, invert=False):
    lo, hi = s.min(), s.max()
    if hi == lo:
        return pd.Series(1.0, index=s.index)
    n = (s - lo) / (hi - lo)
    return 1 - n if invert else n

speed_norm = minmax_norm(composite["inference_ms"], invert=True)
size_norm = minmax_norm(composite["weights_MB"], invert=True)

composite["composite_score"] = (
    0.50 * composite["mAP50-95"]
    + 0.20 * composite["mAP50"]
    + 0.15 * composite["f1"]
    + 0.10 * speed_norm
    + 0.05 * size_norm
)

composite = composite.sort_values("composite_score", ascending=False).reset_index(drop=True)
winner_row = composite.iloc[0]
winner_name = winner_row["model"]

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.barh(composite["model"], composite["composite_score"], color="#8172b3")
bars[0].set_color("#ffd700")  # gold for the winner
ax.invert_yaxis()
ax.set_xlabel("composite score")
ax.set_title("🏆 Final Ranking")
plt.tight_layout()
plt.show()

print(f"🎉🏆 WINNER: {winner_name}  (composite score = {winner_row['composite_score']:.3f}) 🏆🎉")
print(f"   mAP50-95={winner_row['mAP50-95']:.3f} | mAP50={winner_row['mAP50']:.3f} | "
      f"F1={winner_row['f1']:.3f} | {winner_row['inference_ms']:.1f} ms/img | {winner_row['weights_MB']:.1f} MB")

WINNER_MODEL = RUNS[winner_name]["model"]
WINNER_WEIGHTS_PATH = RUNS[winner_name]["weights_path"]

---
## 📌 Step 9: Find Waldo! 🔎 (Fun Finale)

Time to see the winning model in action on a **held-out test set** it has never seen during training.


In [ ]:
TEST_ZIP_PATH = # 📂 Set Your Dataset Path Here
TEST_DIR = # 📂 Set Your Dataset Path Here

if not os.path.exists(TEST_ZIP_PATH):
    print("⬇️  Downloading test set ...")
    !wget -q "https://github.com/kaopanboonyuen/where-is-waldo/raw/main/dataset/waldo-dataset-test.zip" -O "{TEST_ZIP_PATH}"

if not os.path.exists(TEST_DIR):
    print("📦 Extracting test set ...")
    with zipfile.ZipFile(TEST_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall("/content/")

print("✅ Test set ready at", TEST_DIR)

In [ ]:
print(f"🕵️‍♂️ Using the WINNER model: {winner_name}\n   weights: {WINNER_WEIGHTS_PATH}\n")

test_images = sorted(Path(TEST_DIR).glob("*.jpg"))
total_faces = 0

for img_path in test_images:
    img_bgr = cv2.imread(str(img_path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    results = WINNER_MODEL(img_rgb, verbose=False)
    pred = results[0]

    n_found = len(pred.boxes)
    total_faces += n_found

    for box, cls_id in zip(pred.boxes.xyxy.cpu().numpy(), pred.boxes.cls.cpu().numpy()):
        x1, y1, x2, y2 = map(int, box)
        label = pred.names[int(cls_id)]
        cv2.rectangle(img_rgb, (x1, y1), (x2, y2), (0, 255, 0), 3)
        cv2.putText(img_rgb, label, (x1, max(y1 - 8, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    plt.figure(figsize=(9, 9))
    plt.imshow(img_rgb)
    plt.title(f"{img_path.name} — found {n_found} face{'s' if n_found != 1 else ''} 😎")
    plt.axis("off")
    plt.show()

print(f"\n🎉🎉🎉 Total Waldo/Wenda faces found across the test set: {total_faces} 😎")
print(f"🏆 Brought to you by: {winner_name}")

---
## 🎓 Step 10: Mini-Hackathon Assignment (Today's Homework)

### 🎯 Objective

Beat every model on today's scoreboard. Build **your own YOLO experiment(s)** and push the highest `composite_score` you can.

### 📋 Rules

1. Use the **same** `data.yaml` / dataset as today (no extra labeled data).
2. Any Ultralytics YOLO model family/size is allowed (`yolov8*`, `yolo11*`, `yolo12*`, or newer if available).
3. You may tune: architecture size, epochs, image size, augmentation, optimizer, learning rate schedule, or add lightweight tricks (e.g. test-time augmentation, ensembling multiple checkpoints).
4. You may **not** hand-label additional Waldo images or train on the test set — keep it a fair fight.
5. Reuse the notebook's `train_experiment()` / `EXPERIMENTS` / `RUNS` pattern so your run plugs straight into the scoreboard and plots.

### 📦 Deliverables

- Your updated notebook (or a new cell block) with at least **1 new experiment**.
- A screenshot of your entry in the **Scoreboard** table, with its composite score.
- 3–5 sentences: what did you change, and *why* did you think it would help?

### 🏆 Grading Rubric

| Criterion | Weight |
|---|---|
| Model correctness (trains & evaluates without errors) | 30% |
| Improvement over today's best composite score | 40% |
| Clarity of reasoning / EDA-informed decisions | 20% |
| Code cleanliness (reuses the dynamic helpers, no hardcoded paths!) | 10% |

### 💡 Ideas to try

- Bigger backbones (`s`, `m`) vs. more epochs on `n` — which wins under a fixed time budget?
- Since Waldo is *tiny*, try higher `imgsz` (960, 1280) or tiling large images into crops before training.
- Class-imbalance tricks if `waldo` and `wenda` aren't balanced (from your own EDA!).
- Compare optimizers (`SGD` vs `AdamW`) or learning-rate schedules.
- Ensemble two of today's checkpoints and see if it beats either alone.

**Good luck, and may the best detector win! 🕵️‍♂️🏆**


---
### 🙏 Credits

- Dataset & original notebook concept: [kaopanboonyuen/WALDO2026](https://github.com/kaopanboonyuen/WALDO2026)
- Models: [Ultralytics YOLOv8 / YOLO11 / YOLO12 / YOLO2026](https://docs.ultralytics.com/)
- Workshop: NSTDA AI Workshop 2026 — by Dr.Teerapong Panboonyuen (P'Kao)

*Made with 🕵️‍♂️ + ☕ for the next generation of Thai AI builders.*
